# Zirconium spatially resolved cluster dynamics: run a simulation

Irradiated zirconium in one grain whose surface is a grain boundary. At each step MoDELib solves the mobile species (vacancies, interstitials, di- and tri-interstitials) at steady state on quadratic tetrahedra, with the grain boundary at thermal equilibrium. It then advances the loop populations (basal vacancy loops and three families of prismatic interstitial loops) at every finite-element node with CVODE, the mobile field held fixed. The two solves exchange their fields in memory. The equations are those of `docs/reports/GW_Phase4_D1M1.pdf`; the parameters are in `Library/Materials/Zr_CD4opt.txt`.

Everything you normally change is in **§4** and **§5**. Nothing below them needs editing for an ordinary run.

| Section | What it does |
|---|---|
| 1 | Environment: find the MoDELib checkout from this folder |
| 2 | Build: configure and build `DDomp` and `microstructureGenerator` (CMake, SUNDIALS) |
| 3 | The application: what is solved, and the geometries and presets available |
| 4 | Control parameters: preset, doses, steps, integrator |
| 5 | Material and its overrides |
| 6 | Validate: unknown keys and inconsistent settings raise here, before anything runs |
| 7 | Run |
| 8 | Gather the output: volume means, figures, verification checks, report |

Each run writes a new folder `output/<UTC date_time>_<commit>[-dirty]_<tag>/`. On Windows the executables are built and run through WSL; the notebook itself runs with any Python that has NumPy, SciPy and Matplotlib.

## 1. Environment

In [ ]:
# Every location is resolved from this notebook's own folder.
import json, os, sys
from pathlib import Path

os.environ.setdefault("PYTHONIOENCODING", "utf-8")
here = Path.cwd().resolve()
SIMULATIONS = next(p for p in (here, *here.parents, here / "simulations") if (p / "simulation_driver.py").is_file())
if str(SIMULATIONS) not in sys.path:
    sys.path.insert(0, str(SIMULATIONS))

import simulation_driver as driver
import postprocessing

print("MoDELib      :", driver.ROOT)
print("output       :", driver.OUTPUT_DIR)
print("runs through :", "WSL" if driver.USE_WSL else "the local shell")

## 2. Build

Configures once and builds incrementally. SUNDIALS (version 7) is looked for in `$SUNDIALS_ROOT`, in `~/sundials-install`, in a `Libraries/sundials-7.1.1` folder beside the repository, and in the system locations. Without it the build succeeds, and `immobile_integrator="cvode"` is refused in §6.

In [ ]:
FORCE_REBUILD = False      # True discards the build directory and rebuilds from scratch

BUILT = driver.build(force=FORCE_REBUILD)
for name, path in BUILT["executables"].items():
    print(f"{name:>24}: {path}")
print(f"{'SUNDIALS in the build':>24}: {BUILT['sundials']}  ({BUILT['sundials_prefix'] or 'system location'})")

## 3. The application

In [ ]:
geometries = sorted(p.name for p in driver.GEOMETRY_DIR.iterdir() if p.is_dir())
materials = sorted(p.name for p in (driver.LIBRARY / "Materials").glob("Zr*.txt"))
print("geometries :", geometries)
print("presets    :", {name: preset["doses"] for name, preset in driver.PRESETS.items()})
print("materials  :", materials)
print("reference  :", sorted(p.name for p in driver.REFERENCE_DIR.iterdir() if p.is_dir()))

## 4. Control parameters

`PRESET` names a configuration:

- `"reference"`: the cube of 200 nm to 10 dpa, with snapshots at the doses of the ENNDS reference run (1e-4, 1e-3, 1e-2, 0.1, 1, 2, 5, 10 dpa);
- `"hex400"`: the hexagonal crystal of the D1/M1 report to 26 dpa in steps of 1 dpa, with snapshots at 1, 6, 11, 16, 21 and 26 dpa. Its output holds the figure of the loop populations at 6 dpa (Figure 27 of the report) and a comparison with the DisloCluster run of the report;
- `"hex400dc"`: as `"hex400"`, with the two terms of the DisloCluster code that differ from the report; a check of the numerical accuracy against the DisloCluster run.
- `"smoke"`: the cube to 1e-4 dpa, a check of the build and of the workflow.

The dictionaries hold only what you change. A dose interval is covered by `steps_per_interval` steps, or by steps of `step_dpa` when that is given; the mobile species are solved once per step.

In [ ]:
PRESET = os.environ.get("MODELIB_SIMULATION_PRESET", "reference")

CASE = {
    # "geometry": "cube_200nm",
    # "tag": "my_run",                 # last part of the name of the run folder
}
MARCH = {
    # "doses": [1e-4, 1e-3, 1e-2],     # snapshot doses [dpa], increasing
    # "steps_per_interval": 3,
    # "step_dpa": 1.0,               # steps of this dose instead of steps_per_interval
    # "immobile_integrator": "cvode",  # "cvode" (implicit, the D1/M1 model) or "euler" (explicit, as in version 2.1)
}
RUN = {
    # "threads": 8,                    # OpenMP threads; all the cores by default
    # "keep_all_configurations": True, # keep evl_<n>.txt of every step, not only of the snapshots
}
OUTPUT = {
    # "mesh_cut_figures": False,       # skip the 3d/ figures
    # "figure_doses": [1.0, 10.0],     # doses of the 3d/ figures; all the snapshots by default
    # "population_dose": 6.0,          # dose of the figure of the loop populations; the last dose by default
}

## 5. Material and its overrides

`overrides` replaces values of the material file for this run only, by the name of the variable, for example `{"rhoNetwork_SI": "1.0e14", "nNuc": "400 300 300 300"}`. The file in `Library/Materials` is not changed. A name that is not in the file raises in §6.

In [ ]:
MATERIAL = {
    # "material_file": "Zr_CD4opt.txt",
    # "temperature_K": 573.0,          # the reaction prefactors and bias factors of Zr_CD4opt.txt hold at 573 K
    # "dose_rate_dpa_s": 1.0e-7,
    # "overrides": {},
}

## 6. Validate

In [ ]:
CONFIG = driver.configuration(PRESET, CASE=CASE, MATERIAL=MATERIAL, MARCH=MARCH, RUN=RUN, OUTPUT=OUTPUT)
DERIVED = driver.validate(CONFIG, BUILT)

print(json.dumps(CONFIG, indent=1))
print()
print(f"box edge       : {DERIVED['size_b']:.2f} b = {DERIVED['size_b'] * DERIVED['b_SI'] * 1e9:.1f} nm")
print(f"time unit b/cs : {DERIVED['time_unit_s']:.4e} s")
print(f"atomic volume  : {DERIVED['omega_SI']:.4e} m^3")
print(f"steps          : {sum(DERIVED['steps'])} (+1 per interval to write its last configuration)")

## 7. Run

One call of `DDomp` per dose interval. Each call starts from the configuration file written by the previous one: the fields pass from one interval to the next through `evl/evl_<n>.txt`, written at full precision. Within a call, the mobile and immobile solves exchange their fields in memory.

In [ ]:
RUN_DIR = driver.new_run_directory(CONFIG)
print("run folder:", RUN_DIR)
driver.stage(CONFIG, DERIVED, RUN_DIR)
RECORD = driver.run(CONFIG, DERIVED, BUILT, RUN_DIR)
print(RECORD["status"], f"in {RECORD['wall_s']:.0f} s")

## 8. Gather the output

Writes `volume_means.csv`, `summary.json`, `report.md`, `boundary_flux.md` and the figures of `figures/`, `gb/`, `3d/`, `volume_average/`, `size_spectrum/`, `discrete_loops/`, `tem_slices/` and `movies/` into the run folder, then shows the verification checks and some of the figures.

In [ ]:
SUMMARY = postprocessing.gather(RUN_DIR, cuts=CONFIG["OUTPUT"]["mesh_cut_figures"], figure_doses=CONFIG["OUTPUT"]["figure_doses"],
                                samples=CONFIG["OUTPUT"]["voronoi_samples"])
for check in SUMMARY["checks"]:
    print("PASS" if check["passed"] else "FAIL", "|", check["check"], "|", check["result"])

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown((RUN_DIR / "report.md").read_text(encoding="utf-8").split("## Files")[0]))
populations = sorted(p.relative_to(RUN_DIR).as_posix() for p in (RUN_DIR / "figures").glob("loop_populations_*.png"))
for figure in (*populations, "figures/volume_means.png", "figures/comparison_with_ENNDS.png", "figures/comparison_with_reference.png", "volume_average/loop_density.png",
               "volume_average/irradiation_growth.png", "gb/denuded_zone.png", "tem_slices/tem_B_0001_montage.png"):
    if (RUN_DIR / figure).is_file():
        display(Image(filename=str(RUN_DIR / figure)))